In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [14]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [15]:
def transform_super_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [16]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()
    return content

In [17]:
super_dataset = pd.read_csv('../Dataset/super_sms_dataset.csv', encoding='mac_roman')
super_dataset.rename(columns={'Labels': 'label'}, inplace=True)
super_dataset.head()

,SMSes,label
0,There be an update for your delivery CC 017281...,1.0
1,watch your favorite english movies of all genr...,1.0
2,aur what is the status for fms,0.0
3,hi shalini sundi thank you for dialling speci...,1.0
4,m tryin to understand too...,0.0


In [18]:
super_dataset['label'] = super_dataset['label'].replace({1.0:'spam', 0.0:'ham'})

In [19]:
# super_dataset = transform_super_dict(super_dataset)
# super_dataset.head()

In [20]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'super Dataset_'+'.csv')['URL'].to_list())

In [21]:
super_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'super_dataset_'+'.csv')['URL']
super_dataset['Message Len'] = [len(str(i)) for i in super_dataset['SMSes']]
super_dataset.head()

,SMSes,label,Extracted URL,Message Len
0,There be an update for your delivery CC 017281...,spam,http://olo.me/3i6xF,93
1,watch your favorite english movies of all genr...,spam,NaN,146
2,aur what is the status for fms,ham,NaN,30
3,hi shalini sundi thank you for dialling speci...,spam,NaN,112
4,m tryin to understand too...,ham,NaN,28


In [22]:
super_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'super_dataset Websites Analysis'+'.csv')
super_website_analysis_data = super_website_analysis_data.drop(columns=['ham', 'spam'])
super_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,http://50ca13b44a4d.ngrok.io/sbibank,50ca13b44a4d.ngrok.io,0,0,404,0
1,httpysmrtlnkinurvqwr,httpysmrtlnkinurvqwr.,0,0,-1,0
2,httpbitlynysdiy,httpbitlynysdiy.,0,0,-1,0
3,www.imsindia.co,www.imsindia.co,0,0,-1,0
4,httpbitlywfwixn,httpbitlywfwixn.,0,0,-1,0


In [23]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [24]:
super_website_analysis_data.iloc[0][0]

C:\Users\mmia43\AppData\Local\Temp\ipykernel_25012\6995897.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  super_website_analysis_data.iloc[0][0]


'http://50ca13b44a4d.ngrok.io/sbibank'

In [25]:
# for row in super_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [26]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [27]:
extracted_urls = super_dataset['Extracted URL'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(super_website_analysis_data['FQDN'])}
website_data = super_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [28]:
super_dataset['FQDN'] = fqdn
super_dataset['Website Size in KB'] = website_size
super_dataset['Website Textual Content Length'] = text_content_len
super_dataset['Status Code'] = status_code
super_dataset['Parked'] = parked

In [29]:
super_dataset = super_dataset.replace('', np.nan)
super_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_25012\1214194135.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  super_dataset = super_dataset.replace('', np.nan)


,SMSes,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,There be an update for your delivery CC 017281...,spam,http://olo.me/3i6xF,93,olo.me,114.0,0.0,200.0,1.0
1,watch your favorite english movies of all genr...,spam,NaN,146,NaN,NaN,NaN,NaN,NaN
2,aur what is the status for fms,ham,NaN,30,NaN,NaN,NaN,NaN,NaN
3,hi shalini sundi thank you for dialling speci...,spam,NaN,112,NaN,NaN,NaN,NaN,NaN
4,m tryin to understand too...,ham,NaN,28,NaN,NaN,NaN,NaN,NaN


In [30]:
# Counter(super_dataset['FQDN'].to_list())
super_dataset[(super_dataset['Extracted URL'].notna()) & (super_dataset['FQDN'].isna())]

,SMSes,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [31]:
# Counter(super_dataset['FQDN'].to_list())
super_dataset[(super_dataset['Extracted URL'].notna()) & (super_dataset['FQDN'].notna())]

,SMSes,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,There be an update for your delivery CC 017281...,spam,http://olo.me/3i6xF,93,olo.me,114.0,0.0,200.0,1.0
14,Dear SBI user your SBI YONO A/c have be block ...,spam,http://bit.ly/3CloxIZ,155,bit.ly,116249.0,9024.0,200.0,1.0
22,"Good News-Your SBI Credit card of Limit Rs.3,2...",NaN,https://k1s.in,127,k1s.in,605.0,19.0,200.0,0.0
27,To stay connect with the Mercedes Benz family ...,spam,http://bit.ly/2NfDtSn,301,bit.ly,116249.0,9024.0,200.0,1.0
32,buy get free on all yepme men footwear onl...,spam,http://ysmrtlnk.in/vaylpjl,185,ysmrtlnk.in,0.0,0.0,-1.0,0.0
...,...,...,...,...,...,...,...,...,...
66896,For better immunity and to fight against virus...,spam,http://l.jazzmobilemag.pk/y4hD7cn/c8e2,160,l.jazzmobilemag.pk,0.0,0.0,-1.0,0.0
66897,UPTO Rs X LAKH OFF On Popular Used Cars India ...,spam,http://goo.gl/N8uSjc,144,goo.gl,67934.0,5090.0,200.0,1.0
66910,Dear Deeksha Singh Congratulations You have su...,spam,http://cowin.gov.in,172,cowin.gov.in,19910.0,0.0,200.0,1.0
66961,Scotiabank Alert Your account be disabled due ...,spam,http://scotiabank.ca.ssl-12-23-102,149,scotiabank.ca.ssl-12-23-102.,0.0,0.0,-1.0,0.0


In [32]:
print(len(super_dataset))

67010


In [33]:
#messages with URL
print(len(super_dataset[(super_dataset['Extracted URL'].notna())]), len(super_dataset[(super_dataset['Extracted URL'].notna())])/len(super_dataset))

5283 0.07883897925682734


In [34]:
#spam messages with URL
print(len(super_dataset[(super_dataset['Extracted URL'].notna()) & (super_dataset['label']=='spam')]), len(super_dataset[(super_dataset['Extracted URL'].notna()) & (super_dataset['label']=='spam')])/len(super_dataset[super_dataset['label']=='spam']))

5265 0.2011230804492322


In [35]:
#unique FQDN
len(set(super_dataset[(super_dataset['FQDN'].notna())]['FQDN']))

2539

In [36]:
only_unique_live_websites_data = super_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

#live websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['label']=='ham')]))

#live websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['label']=='spam')]))

476
4
471


In [37]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

#parked websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['label']=='ham')]))

#parked websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['label']=='spam')]))

126
1
125


In [38]:
super_dataset.to_csv('../Dataset/Refined_Super_Dataset.csv', index=None)